### Extração de dados pelo BrGaap

### Extração usando método em páginas

considere classificar por individual e consolidado caso seja obrigatório

In [29]:
import pdfplumber
import pandas as pd
import re
import numpy as np
import re
from datetime import datetime

In [30]:

with pdfplumber.open("BrGaap - Demonstrações Contábeis 3T25.pdf") as pdf:
    paginas = pdf.pages[31:36]  # intervalo desejado

    textos = []
    for page in paginas:
        txt = page.extract_text()
        if txt:
            textos.append(txt)

# Junta todas as páginas
text = "\n".join(textos)

# Pré-processamento
text = "\n".join(line.strip() for line in text.splitlines() if line.strip())

# Define início e fim
start = text.find("(b) Movimentação por estágios da carteira de crédito")
end = text.find("(c) Composição da carteira por faixa de atraso", start)

trecho = text[start:end]

print(trecho)

linhas = [l.strip() for l in trecho.splitlines() if l.strip()]


(b) Movimentação por estágios da carteira de crédito
Individual
Saldo em Constituição/ Transferência do/ Transferência do/ Saldo em
Estágio 1 (1)
01/01/2025 (liquidação) para o estágio 2 para estágio 3 30/09/2025
Empréstimos e direitos creditórios descontados 168.993.688 18.644.030 8.278.831 (4.527.185) 191.389.364
Financiamentos 7.910.356 3.520.672 (77.807) (265.104) 11.088.117
Financiamentos rurais 57.442.005 (1.418.017) (3.165.185) (3.741.667) 49.117.136
Financiamentos imobiliários 786.474.231 73.081.011 (5.827.539) (6.642.937) 847.084.766
Financiamentos de infraestrutura 100.829.120 3.755.744 (130.086) (144.896) 104.309.882
Cessão de crédito 3.293.659 (394.364) (13.987) (19.012) 2.866.296
Outros ativos com características de concessão de crédito (2) 16.078.584 (13.667.447) (4.216) (220.146) 2.186.775
Total 1.141.021.643 83.521.629 (939.989) (15.560.947) 1.208.042.336
(1) Inclui o montante de R$ 39.461.483 referente aos contratos com mais de 30 dias de atraso.
(2) Movimentação consi

In [31]:
PRODUTOS = [
    "Financiamentos",
    "Financiamentos rurais",
    "Financiamentos imobiliários",
    "Financiamentos de infraestrutura",
    "Cessão de crédito",
    "Outros ativos com características de concessão de crédito",
    "Total"
]

def eh_produto(linha: str) -> bool:
    return any(linha.startswith(p) for p in PRODUTOS)


# Código para evitar que o código só leia a primeira palavra de PRODUTOS
PRODUTOS_ORDENADOS = sorted(PRODUTOS, key=len, reverse=True)

# apenas o primeiro e o último valor
def reduzir_linha_produto(linha: str) -> str:
    partes = linha.split()
    
    # pega apenas tokens que parecem números (financeiros)
    numeros = [
        p for p in partes
        if re.search(r"\d", p)  # contém ao menos um dígito
    ]
    
    primeiro_valor = numeros[0]
    ultimo_valor = numeros[-1]

    # identifica o produto (nome composto)
    for produto in PRODUTOS_ORDENADOS:
        if linha.startswith(produto):
            return f"{produto} {primeiro_valor} {ultimo_valor}"

    return linha


# regras de limpeza de dados
def reduzir_texto(texto: str) -> str:
    linhas = [l.strip() for l in texto.splitlines() if l.strip()]
    saida = []

    for linha in linhas:
        
        # eu posso substituir todos os ifs de cada fórmula por uma condição dessas
        if not any(linha.startswith(p) for p in PRODUTOS) and not linha.startswith("Estágio") and not linha.startswith("Total") and not linha.startswith("Individual") and not linha.startswith("Consolidado"):
            continue
        
        # faz um tratamento no estágio 1
        if linha.startswith("Estágio 1 (1)"):
            linha = linha.replace("Estágio 1 (1)", "Estágio 1")
        
        # Redução dos produtos
        if eh_produto(linha):
            saida.append(reduzir_linha_produto(linha))
        else:
            saida.append(linha)
            
    return "\n".join(saida)


In [39]:
texto = reduzir_texto(trecho)
print(texto)

Individual
Estágio 1
Financiamentos 7.910.356 11.088.117
Financiamentos rurais 57.442.005 49.117.136
Financiamentos imobiliários 786.474.231 847.084.766
Financiamentos de infraestrutura 100.829.120 104.309.882
Cessão de crédito 3.293.659 2.866.296
Outros ativos com características de concessão de crédito (2) 2.186.775
Total 1.141.021.643 1.208.042.336
Individual
Estágio 2
Financiamentos 437.628 522.966
Financiamentos rurais 850.510 4.137.789
Financiamentos imobiliários 6.118.306 11.187.917
Financiamentos de infraestrutura 453.491 545.942
Cessão de crédito 26.741 33.207
Outros ativos com características de concessão de crédito 1.258.829 8.485
Total 30.433.911 27.705.353
Individual
Estágio 3
Financiamentos 578.823 1.419.626
Financiamentos rurais 4.008.564 7.742.501
Financiamentos imobiliários 32.603.630 37.999.082
Financiamentos de infraestrutura 5.865.448 5.757.808
Cessão de crédito 76.597 91.926
Outros ativos com características de concessão de crédito 523.548 310.925
Total 65.401.947 

### Extração de datas

In [33]:
# Extração de anos e meses
# para criar a coluna de trimestre

from datetime import datetime

padrao_data = re.compile(r"\b\d{2}/\d{2}/\d{4}\b")
datas = padrao_data.findall(trecho)

print(datas)

def trimestre_from_date(date_str: str) -> str:
    dt = datetime.strptime(date_str, "%d/%m/%Y")
    trimestre_map = {1: "1T", 3: "1T", 6: "2T", 9: "3T", 12: "4T"}
    trimestre = trimestre_map.get(dt.month)

    if not trimestre:
        raise ValueError(f"Mês inesperado na data: {date_str}")

    return f"{trimestre}{str(dt.year)[-2:]}"

def extract_anos(texto: str) -> list[str]:
    datas = re.findall(r"\d{2}/\d{2}/\d{4}", texto)

    if len(datas) < 2:
        raise ValueError("Não foi possível encontrar duas datas finais")
    
    datas_finais = datas[-2:]
    return [trimestre_from_date(d) for d in datas_finais]
    

['01/01/2025', '30/09/2025', '01/01/2025', '30/09/2025', '01/01/2025', '30/09/2025', '01/01/2025', '30/09/2025', '01/01/2025', '30/09/2025', '01/01/2025', '30/09/2025']


In [34]:
extract_anos(trecho)

['1T25', '3T25']

### Individual x Consolidado
É o que vai definir se o valor que será extraído será com as células dentro de "Individual" ou "Consolidado"

In [ ]:
def categorias(texto: str, categoria_escolhida: str):
    resultado = []
    categoria_atual = None

    linhas = [l.strip() for l in texto.splitlines() if l.strip()]

    for linha in linhas:

        # Detecta categoria (case insensitive)
        if re.fullmatch(r"Individual", linha, flags=re.IGNORECASE):
            categoria_atual = "Individual"
            continue

        elif re.fullmatch(r"Consolidado", linha, flags=re.IGNORECASE):
            categoria_atual = "Consolidado"
            continue

        # Só captura se estiver dentro da categoria escolhida
        if categoria_atual == categoria_escolhida:
            resultado.append(linha)

    return resultado


print("\n".join(categorias(texto, "Individual")))



Estágio 1
Financiamentos 7.910.356 11.088.117
Financiamentos rurais 57.442.005 49.117.136
Financiamentos imobiliários 786.474.231 847.084.766
Financiamentos de infraestrutura 100.829.120 104.309.882
Cessão de crédito 3.293.659 2.866.296
Outros ativos com características de concessão de crédito (2) 2.186.775
Total 1.141.021.643 1.208.042.336
Estágio 2
Financiamentos 437.628 522.966
Financiamentos rurais 850.510 4.137.789
Financiamentos imobiliários 6.118.306 11.187.917
Financiamentos de infraestrutura 453.491 545.942
Cessão de crédito 26.741 33.207
Outros ativos com características de concessão de crédito 1.258.829 8.485
Total 30.433.911 27.705.353
Estágio 3
Financiamentos 578.823 1.419.626
Financiamentos rurais 4.008.564 7.742.501
Financiamentos imobiliários 32.603.630 37.999.082
Financiamentos de infraestrutura 5.865.448 5.757.808
Cessão de crédito 76.597 91.926
Outros ativos com características de concessão de crédito 523.548 310.925
Total 65.401.947 81.394.304


### Blocagem de estágios

In [59]:
def split_blocos_estagio(texto: str) -> dict:
    blocos = {}
    estagio_atual = None
    buffer = []

    for linha in texto.splitlines():
        
        match = re.match(r"Estágio\s+(\d+)", linha)
        if match:
            # salva bloco anterior
            if estagio_atual is not None:
                blocos[estagio_atual] = "\n".join(buffer)
                buffer = []
            
            estagio_atual = int(match.group(1))
            continue
        
        if estagio_atual is not None:
            buffer.append(linha)

    # salva último bloco
    if estagio_atual is not None:
        blocos[estagio_atual] = "\n".join(buffer)

    return blocos


In [60]:
def parse_linha_produto(linha: str):
    for produto in sorted(PRODUTOS, key=len, reverse=True):
        if linha.startswith(produto):
            
            valores = linha[len(produto):].strip().split()
            
            return produto, valores

    return None, []


### Tratamento de formato do número

In [65]:
def normalizar_valor(valor: str):
    if valor == "—":
        return None

    # remove espaços
    valor = valor.strip()

    # caso (123) → -123
    if re.fullmatch(r"\(\d+\)", valor):
        valor = "-" + valor[1:-1]

    # remove separador de milhar
    valor = valor.replace(".", "")

    return int(float(valor))

### Criação de um Dataframe

In [66]:
def construir_linhas_caixa(texto_reduzido: str, trecho_original: str) -> list[dict]:
    
    anos = extract_anos(trecho_original)
    blocos = split_blocos_estagio(texto_reduzido)

    linhas_finais = []

    for estagio, bloco in blocos.items():
        
        for linha in bloco.splitlines():
            
            if eh_produto(linha):
                
                produto, valores = parse_linha_produto(linha)

                for idx, ano in enumerate(anos):
                    
                    valor = (
                        normalizar_valor(valores[idx])
                        if idx < len(valores)
                        else None
                    )

                    linhas_finais.append({
                        "ano": ano,
                        "banco": "Caixa",
                        "produto": produto,
                        "PD": "-",
                        "Estágio": estagio,
                        "Exposição Bruta": valor,
                    })

    return linhas_finais


In [ ]:
texto_reduzido = reduzir_texto(trecho)
texto_individual = "\n".join(categorias(texto_reduzido, "Individual"))

linhas = construir_linhas_caixa(texto_individual, trecho)

df_final = pd.DataFrame(linhas)

# remove total
df_final = df_final[df_final["produto"] != "Total"].copy()
df_final



,ano,banco,produto,PD,Estágio,Exposição Bruta
0,1T25,Caixa,Financiamentos,-,1,7910356
1,3T25,Caixa,Financiamentos,-,1,11088117
2,1T25,Caixa,Financiamentos rurais,-,1,57442005
3,3T25,Caixa,Financiamentos rurais,-,1,49117136
4,1T25,Caixa,Financiamentos imobiliários,-,1,786474231
5,3T25,Caixa,Financiamentos imobiliários,-,1,847084766
6,1T25,Caixa,Financiamentos de infraestrutura,-,1,100829120
7,3T25,Caixa,Financiamentos de infraestrutura,-,1,104309882
8,1T25,Caixa,Cessão de crédito,-,1,3293659
9,3T25,Caixa,Cessão de crédito,-,1,2866296


### Extração de dados pelo IFRS ( aqui tem o perdas esperadas )

In [ ]:
# Ingerir PDF
import pdfplumber
import pandas as pd
import re
import numpy as np
import re

In [ ]:
with pdfplumber.open("Demonstrações contábeis consolidadas Caixa 2T25.pdf") as pdf:
    page = pdf.pages[51]   # página 53 (índice começa em 0)
    text = page.extract_text()


# Remove espaços em branco extras de cada linha ( pré-processamento )
text = "\n".join(line.strip() for line in text.splitlines())

# Define início ( start ) e fim ( end ) do trecho a ser extraído. 
start = text.find("(a) Movimentação da provisão para perdas esperadas")
end = text.find("(b) Movimentação da provisão para perdas esperadas", start)

trecho = text[start:end]
print(trecho)

# divide em linhas para facilitar a criação do dataframe
linhas = [l.strip() for l in trecho.splitlines() if l.strip()]

In [ ]:
# Qual será o formato da tabela, neste caso?